# Adverse Competition in Concentrated Liquidity AMMs

## Evidence from Uniswap V3

### Research Question

In concentrated liquidity AMMs, large liquidity providers (LPs) can concentrate capital around the active tick, capturing a disproportionate share of fee revenue. We term this **adverse competition** — an LP-vs-LP dynamic distinct from **adverse selection** (LVR), which is a trader-vs-LP dynamic.

We construct a **congestion index** $\Delta I_t$ from a structural state-space model that captures LP repositioning unexplained by market conditions. We then test whether $\Delta I_t$ has a negative, statistically significant impact on fee-adjusted returns, **orthogonal to LVR**.

### Two-Stage Estimation

**Stage 1 — Extract congestion index:**

$$\frac{\Delta L_t}{L_{t-1}} = \beta_1 \frac{\Delta P_t}{P_{t-1}} + \beta_2 \cdot \text{txActivity}_t + e_t, \quad e_t = \gamma e_{t-1} + v_t$$

$$\Delta I_t \equiv e_t$$

**Stage 2 — Test adverse competition impact (orthogonal to LVR):**

$$\Delta \text{feeYield}_t = \alpha + \delta_1 \left|\frac{\Delta P_t}{P_{t-1}}\right| + \eta_t \quad \text{(strip LVR)}$$

$$\eta_t = \mu + \delta_2 \cdot \Delta I_t + \varepsilon_t \quad \text{(HC1 robust SEs)}$$

**Key result:** $\delta_2 = -0.002$, $z = -3.74$, $p < 0.001$ on 1,731 daily observations.

### Connection to Product Design

$\Delta I_t$ is the state variable that drives the **congestionToken** sigmoid pricing function $p(I) = \sigma(I/\lambda)$. The statistical significance of $\delta_2$ validates the economic mechanism — congestion creates measurable fee compression — that generates hedging demand for the instrument.

## 1. Data

**Pool:** Uniswap V3 USDC/WETH (`0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640`), 5 bps fee tier.

**Sample:** 1,760 daily observations (pool lifetime), ~11M transactions.

**Variables:**
| Variable | Definition |
|---|---|
| $\text{tvlUSD}_t$ | Total value locked (USD) |
| $\text{volumeUSD}_t$ | Daily trading volume (USD) |
| $\text{feesUSD}_t$ | Daily fee revenue (USD) |
| $P_t$ | token0 price (USDC per WETH) |
| $\text{txCount}_t$ | Daily transaction count |

**Derived series:**
| Series | Construction |
|---|---|
| $\Delta L_t / L_{t-1}$ | `delta(tvlUSD) / lagged(tvlUSD)` |
| $\Delta P_t / P_{t-1}$ | `delta(priceUSD) / lagged(priceUSD)` |
| $\text{txActivity}_t$ | `txCount / rolling_mean(txCount, 30)` |
| $\text{feeYield}_t$ | `feesUSD / tvlUSD` |
| $\lvert\Delta P / P\rvert$ | Absolute price returns (LVR proxy) |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from data.DataHandler import (
    PoolEntryData, delta, tvlUSD, priceUSD, volumeUSD, feesUSD,
    div, lagged, txCount, normalize
)
from data.Econometrics import (
    LiquidityStateModel, AdverseCompetitionModel,
    beta, rho, state, result, delta_coeff, residual, ols_result
)
from data.UniswapClient import UniswapClient, v3

# ── Plotly monochrome template ──────────────────────────────────
classic = go.layout.Template()
classic.layout = go.Layout(
    font=dict(family="Courier New, monospace", size=12, color="#1a1a1a"),
    paper_bgcolor="#fafaf5",
    plot_bgcolor="#fafaf5",
    title=dict(font=dict(size=16, family="Courier New, monospace")),
    xaxis=dict(
        showgrid=True, gridcolor="#cccccc", gridwidth=0.5,
        linecolor="#1a1a1a", linewidth=1, mirror=True,
        zeroline=True, zerolinecolor="#999999", zerolinewidth=0.8
    ),
    yaxis=dict(
        showgrid=True, gridcolor="#cccccc", gridwidth=0.5,
        linecolor="#1a1a1a", linewidth=1, mirror=True,
        zeroline=True, zerolinecolor="#999999", zerolinewidth=0.8
    ),
    colorway=["#1a1a1a", "#666666", "#999999", "#bbbbbb"],
)
pio.templates["classic"] = classic
pio.templates.default = "classic"

# ── Load data ───────────────────────────────────────────────────
V3_USDC_WETH = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
client = UniswapClient(v3())
pool = PoolEntryData(V3_USDC_WETH, client=client)
pool_data = pool(pool.lifetimeLen())

# ── Summary statistics ──────────────────────────────────────────
summary = pool_data[["tvlUSD", "volumeUSD", "feesUSD", "token0Price", "txCount"]].describe()
summary.columns = ["TVL (USD)", "Volume (USD)", "Fees (USD)", "Price (USDC/WETH)", "Tx Count"]
print(f"Pool: V3 USDC/WETH 5bps")
print(f"Observations: {len(pool_data)}")
print(f"Period: {pool_data.index[0].date()} to {pool_data.index[-1].date()}")
print()
summary.round(2)